In [88]:
import xmltodict
from pprint import pprint

In [89]:
def minimize_nesting(data):
    """
    Minimize the nesting of certain fields in the item dictionary.

    Args:
        item: Dictionary containing entity data
    """
    field_mapping = [
        ("names", "name"),
        ("addresses", "address"),
        ("features", "feature"),
        ("relationships", "relationship"),
        ("legalAuthorities", "legalAuthority"),
        ("sanctionsLists", "sanctionsList"),
        ("sanctionsPrograms", "sanctionsProgram"),
        ("sanctionsTypes", "sanctionsType"),
        ("translations", "translation"),
        ("nameParts", "namePart"),
        ("addressParts", "addressPart")
    ]

    for root, child in field_mapping:
        if root in data:
            try:
                # Handle case where root exists but child might not
                nested_data = data[root].get(child)
                if nested_data is not None:
                    # Convert to list if single item
                    data[root] = (
                        nested_data if isinstance(nested_data, list) else [nested_data]
                    )
            except (AttributeError, TypeError):
                # Keep original data if structure is not as expected
                continue

    # Recursively process nested dictionaries
    for key, value in data.items():
        if isinstance(value, dict):
            data[key] = minimize_nesting(value)
        elif isinstance(value, list):
            data[key] = [
                minimize_nesting(item) if isinstance(item, dict) else item
                for item in value
            ]

    return data

In [90]:
def replace_text_objects(data):
    """
    Recursively search through nested dictionary/list and replace objects
    containing '#text' with their text value

    Args:
        data: Input dictionary or list to process

    Returns:
        Modified data structure with '#text' objects replaced by their values
    """
    if isinstance(data, dict):
        if "#text" in data:
            return data["#text"]

        return {key: replace_text_objects(value) for key, value in data.items()}

    elif isinstance(data, list):
        return [replace_text_objects(item) for item in data]

    return data

In [91]:
xml = open('2025-06-10_delta.xml', 'r').read()

In [92]:
data = xmltodict.parse(xml)

In [93]:
for item in data["sanctionsData"]["entities"]["entity"]:
    item = replace_text_objects(item)
    item = minimize_nesting(item)
    pprint(item)

{'@action': 'add',
 '@id': '53912',
 'addresses': [{'@id': '82267',
                'country': 'Algeria',
                'translations': [{'@id': '64283',
                                  'addressParts': [{'@id': '113966',
                                                    'type': 'CITY',
                                                    'value': 'Tizi Ouzou'}],
                                  'isPrimary': 'true',
                                  'script': 'Latin'}]},
               {'@id': '82268',
                'country': 'Algeria',
                'translations': [{'@id': '64284',
                                  'addressParts': [{'@id': '113967',
                                                    'type': 'CITY',
                                                    'value': 'Algiers'}],
                                  'isPrimary': 'true',
                                  'script': 'Latin'}]}],
 'features': [{'@id': '90406',
               'isPrimary': 'true',
         